# VERIDEX — Fold 5 Collapse Diagnostic

**Purpose:** Find out exactly which properties are causing fold 5 to collapse.

**This notebook does NOT change anything in your pipeline.**
It only reads from the database and prints diagnostic information.

Run every cell top to bottom in order.

## Cell 1 — Imports and Setup

In [1]:
import sqlite3
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score

DB_PATH = "florida_leads.db"

FEATURE_COLUMNS = [
    "JV", "ACT_YR_BLT", "SALE_PRC1", "SALE_YR1",
    "LND_SQFOOT", "TOT_LVG_AR", "NO_BULDNG",
    "SALE_PRC2", "SALE_YR2", "NO_RES_UNT",
]

TARGET_COLUMN = "hot_lead"

print("Setup complete.")

Setup complete.


## Cell 2 — Load Data From Database

In [2]:
conn = sqlite3.connect(DB_PATH)
df   = pd.read_sql_query("SELECT * FROM leads_features", conn)
conn.close()

print(f"Loaded : {len(df):,} records")
print(f"Columns: {len(df.columns)}")
print(f"Hot leads  : {df['hot_lead'].sum():,} ({df['hot_lead'].mean()*100:.1f}%)")
print(f"Cold leads : {(df['hot_lead']==0).sum():,}")

Loaded : 34,606 records
Columns: 39
Hot leads  : 4,984 (14.4%)
Cold leads : 29,622


## Cell 3 — Prepare X and y

In [3]:
X = df[FEATURE_COLUMNS].fillna(0).values
y = df[TARGET_COLUMN].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print("Scaling done.")

X shape : (34606, 10)
y shape : (34606,)
Scaling done.


## Cell 4 — Run Each Fold and Print Full Diagnostics

This is the main diagnostic cell.
For each fold it prints:
- F1 score
- Hot lead percentage in that fold's test set
- City distribution
- DOR_UC (property type) distribution
- Neighbourhood distribution
- JV value range
- SALE_YR1 range (how old the sales are)

The fold that looks **different** from the others is the problem.

In [4]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("=" * 65)
print("FOLD-BY-FOLD DIAGNOSTIC — VERIDEX")
print("=" * 65)

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y), start=1):

    X_tr, X_te = X_scaled[train_idx], X_scaled[test_idx]
    y_tr, y_te = y[train_idx],        y[test_idx]

    # ── Train a quick model on this fold ──────────────────────
    model = GradientBoostingClassifier(
        n_estimators  = 100,
        max_depth     = 4,
        learning_rate = 0.05,
        random_state  = 42
    )
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)

    f1        = f1_score(y_te, y_pred)
    precision = precision_score(y_te, y_pred)
    recall    = recall_score(y_te, y_pred)

    # ── Pull the actual property records for this test fold ───
    fold_df = df.iloc[test_idx].copy()

    # Hot lead distribution
    hot_count = int(y_te.sum())
    hot_pct   = hot_count / len(y_te) * 100

    # City distribution
    city_dist = fold_df["PHY_CITY"].value_counts().head(5).to_dict()

    # DOR_UC distribution (property type)
    dor_dist  = fold_df["DOR_UC"].value_counts().to_dict()

    # Neighbourhood distribution — top 5
    nbrhd_dist = fold_df["NBRHD_CD"].value_counts().head(5).to_dict()

    # JV range in this fold
    jv_min  = fold_df["JV"].min()
    jv_max  = fold_df["JV"].max()
    jv_mean = fold_df["JV"].mean()

    # Sale year range
    sale_yr_min  = fold_df["SALE_YR1"].min()
    sale_yr_max  = fold_df["SALE_YR1"].max()

    # Value tier distribution — shows which price brackets land here
    if "value_tier" in fold_df.columns:
        tier_dist = fold_df["value_tier"].value_counts().to_dict()
    else:
        tier_dist = "not available"

    # ── Print everything ──────────────────────────────────────
    marker = "  ← COLLAPSE" if f1 < 0.65 else ""

    print(f"\nFold {fold_num}{marker}")
    print(f"  ─── Model Performance ───────────────────────────")
    print(f"  F1        : {f1:.3f}")
    print(f"  Precision : {precision:.3f}")
    print(f"  Recall    : {recall:.3f}")
    print(f"  ─── Hot Lead Distribution ───────────────────────")
    print(f"  Test size : {len(y_te):,}")
    print(f"  Hot leads : {hot_count:,} ({hot_pct:.1f}%)")
    print(f"  Cold      : {len(y_te) - hot_count:,}")
    print(f"  ─── Property Characteristics ────────────────────")
    print(f"  JV range  : ${jv_min:,.0f} — ${jv_max:,.0f} | mean ${jv_mean:,.0f}")
    print(f"  Sale year : {sale_yr_min:.0f} — {sale_yr_max:.0f}")
    print(f"  ─── Geographic / Type Distribution ──────────────")
    print(f"  Cities    : {city_dist}")
    print(f"  DOR_UC    : {dor_dist}")
    print(f"  NBRHD_CD  : {nbrhd_dist}")
    print(f"  Value tier: {tier_dist}")

print("\n" + "=" * 65)
print("Look at the fold marked ← COLLAPSE and compare it")
print("against the other folds. What is different about it?")
print("=" * 65)

FOLD-BY-FOLD DIAGNOSTIC — VERIDEX

Fold 1
  ─── Model Performance ───────────────────────────
  F1        : 0.754
  Precision : 0.847
  Recall    : 0.679
  ─── Hot Lead Distribution ───────────────────────
  Test size : 6,922
  Hot leads : 997 (14.4%)
  Cold      : 5,925
  ─── Property Characteristics ────────────────────
  JV range  : $1,884 — $562,930 | mean $262,753
  Sale year : 0 — 2026
  ─── Geographic / Type Distribution ──────────────
  Cities    : {'GAINESVILLE': 2377, 'SUWANNEE': 1774, "ST. JOHN'S": 1026, 'ALACHUA': 703, 'HIGH SPRINGS': 529}
  DOR_UC    : {'001': 6165, '002': 616, '010': 73, '007': 68}
  NBRHD_CD  : {'154200.00': 111, '154324.10': 104, '215100.90': 99, '233200.00': 93, '114335.12': 83}
  Value tier: {'mid': 3781, 'upper': 2093, 'entry': 770, 'premium': 165, 'distressed': 113}

Fold 2
  ─── Model Performance ───────────────────────────
  F1        : 0.757
  Precision : 0.855
  Recall    : 0.679
  ─── Hot Lead Distribution ───────────────────────
  Test size : 

## Cell 5 — Deep Dive: What Properties Are In The Collapsed Fold?

This cell takes the worst-performing fold and shows you
which specific properties the model got wrong.

It shows you:
- Properties the model called HOT but were actually COLD (false alarms)
- Properties the model called COLD but were actually HOT (missed leads)

In [6]:
# Find the worst fold automatically
fold_f1s = []

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y), start=1):
    X_tr, X_te = X_scaled[train_idx], X_scaled[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    model = GradientBoostingClassifier(
        n_estimators=100, max_depth=4,
        learning_rate=0.05, random_state=42
    )
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    fold_f1s.append((f1_score(y_te, y_pred), test_idx, y_te, y_pred))

# Get the worst fold
worst_f1, worst_test_idx, worst_y_te, worst_y_pred = min(fold_f1s, key=lambda x: x[0])
worst_fold_num = fold_f1s.index(min(fold_f1s, key=lambda x: x[0])) + 1

print(f"Worst fold: Fold {worst_fold_num} | F1 = {worst_f1:.3f}")
print()

worst_df = df.iloc[worst_test_idx].copy()
worst_df["predicted"] = worst_y_pred
worst_df["actual"]    = worst_y_te

# False Negatives — real hot leads the model MISSED
missed = worst_df[
    (worst_df["actual"] == 1) & (worst_df["predicted"] == 0)
]

# False Positives — cold properties the model wrongly flagged
false_alarms = worst_df[
    (worst_df["actual"] == 0) & (worst_df["predicted"] == 1)
]

print(f"Real hot leads MISSED by model   : {len(missed):,}")
print(f"False alarms (wrongly flagged)   : {len(false_alarms):,}")
print()

# What do the MISSED hot leads look like?
print("─" * 55)
print("MISSED HOT LEADS — Key Characteristics")
print("─" * 55)
print(f"  Count          : {len(missed):,}")

if len(missed) > 0:
    print(f"  JV mean        : ${missed['JV'].mean():,.0f}")
    print(f"  JV range       : ${missed['JV'].min():,.0f} — ${missed['JV'].max():,.0f}")
    print(f"  ACT_YR_BLT mean: {missed['ACT_YR_BLT'].mean():.0f}")
    print(f"  SALE_YR1 mean  : {missed['SALE_YR1'].mean():.0f}")
    if "PHY_CITY" in missed.columns:
        print(f"  Top cities     : {missed['PHY_CITY'].value_counts().head(3).to_dict()}")
    if "value_tier" in missed.columns:
        print(f"  Value tiers    : {missed['value_tier'].value_counts().to_dict()}")
    if "DOR_UC" in missed.columns:
        print(f"  DOR_UC         : {missed['DOR_UC'].value_counts().to_dict()}")

print()
print("─" * 55)
print("FALSE ALARMS — Key Characteristics")
print("─" * 55)
print(f"  Count          : {len(false_alarms):,}")

if len(false_alarms) > 0:
    print(f"  JV mean        : ${false_alarms['JV'].mean():,.0f}")
    print(f"  JV range       : ${false_alarms['JV'].min():,.0f} — ${false_alarms['JV'].max():,.0f}")
    print(f"  ACT_YR_BLT mean: {false_alarms['ACT_YR_BLT'].mean():.0f}")
    print(f"  SALE_YR1 mean  : {false_alarms['SALE_YR1'].mean():.0f}")
    if "PHY_CITY" in false_alarms.columns:
        print(f"  Top cities     : {false_alarms['PHY_CITY'].value_counts().head(3).to_dict()}")
    if "value_tier" in false_alarms.columns:
        print(f"  Value tiers    : {false_alarms['value_tier'].value_counts().to_dict()}")
    if "DOR_UC" in false_alarms.columns:
        print(f"  DOR_UC         : {false_alarms['DOR_UC'].value_counts().to_dict()}")

Worst fold: Fold 1 | F1 = 0.754

Real hot leads MISSED by model   : 320
False alarms (wrongly flagged)   : 122

───────────────────────────────────────────────────────
MISSED HOT LEADS — Key Characteristics
───────────────────────────────────────────────────────
  Count          : 320
  JV mean        : $217,054
  JV range       : $71,699 — $550,837
  ACT_YR_BLT mean: 1978
  SALE_YR1 mean  : 1974
  Top cities     : {'GAINESVILLE': 121, 'SUWANNEE': 95, "ST. JOHN'S": 35}
  Value tiers    : {'mid': 229, 'entry': 49, 'upper': 40, 'premium': 2}
  DOR_UC         : {'001': 257, '002': 53, '007': 6, '010': 4}

───────────────────────────────────────────────────────
FALSE ALARMS — Key Characteristics
───────────────────────────────────────────────────────
  Count          : 122
  JV mean        : $135,867
  JV range       : $49,979 — $425,037
  ACT_YR_BLT mean: 1975
  SALE_YR1 mean  : 1918
  Top cities     : {'SUWANNEE': 36, 'GAINESVILLE': 25, 'HIGH SPRINGS': 20}
  Value tiers    : {'entry': 89

## Cell 6 — Signal Score Distribution Per Fold

This checks whether one fold has a very different
distribution of signal scores (0-7) compared to the others.

If fold 5 has a very different signal score pattern,
it means the hot_lead label itself is inconsistent
across different slices of your data.

In [7]:
print("=" * 65)
print("HOT LEAD % PER FOLD — should be roughly equal across all folds")
print("If one fold is very different, the label has inconsistency.")
print("=" * 65)

for fold_num, (train_idx, test_idx) in enumerate(skf.split(X_scaled, y), start=1):
    y_te      = y[test_idx]
    fold_df   = df.iloc[test_idx]
    hot_pct   = y_te.mean() * 100
    jv_mean   = fold_df["JV"].mean()
    city_top  = fold_df["PHY_CITY"].value_counts().index[0] if "PHY_CITY" in fold_df.columns else "N/A"

    print(f"  Fold {fold_num} | hot={hot_pct:.1f}% | JV mean=${jv_mean:,.0f} | top city={city_top}")

print()
print("If hot% is similar across all folds → label is consistent")
print("If one fold has very different hot% → that fold has unusual properties")

HOT LEAD % PER FOLD — should be roughly equal across all folds
If one fold is very different, the label has inconsistency.
  Fold 1 | hot=14.4% | JV mean=$262,753 | top city=GAINESVILLE
  Fold 2 | hot=14.4% | JV mean=$263,388 | top city=GAINESVILLE
  Fold 3 | hot=14.4% | JV mean=$263,665 | top city=GAINESVILLE
  Fold 4 | hot=14.4% | JV mean=$263,129 | top city=GAINESVILLE
  Fold 5 | hot=14.4% | JV mean=$266,276 | top city=GAINESVILLE

If hot% is similar across all folds → label is consistent
If one fold has very different hot% → that fold has unusual properties
